# 📐 Module 3 — Class 2 Homework: Scaling

**Topic:** Scaling — StandardScaler, MinMaxScaler, RobustScaler

## 📋 What you have to do

This notebook **already has working code**. Your job:

1. **Run every cell from top to bottom** — it should run with no errors.
2. **Add a comment on EVERY code cell** that explains what the cell does, in your own words.
   - Use `#` to write comments inside the code cell.
   - Write at least 1 short sentence per cell. Longer is better.
3. **Save your notebook** as `Module<X>_Class<Y>_<YourName>.ipynb` and submit.

**Example of a good commented cell:**

```python
# Count how many customers churned vs stayed
# value_counts() returns the number of rows for each unique value
df['Churn'].value_counts()
```

**Example of a BAD comment (do not do this):**

```python
# count
df['Churn'].value_counts()
```

---


In [1]:
# === SETUP — run this first ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)  # Convert TotalCharges from string to float, forcing blank spaces to NaN, and fill missing values with 0
print('Loaded:', df.shape)


Loaded: (7043, 21)


### Cell 1 — see how different the ranges are

In [2]:
# 1. Define the specific key numeric columns to analyze
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# 2. Get descriptive statistics (mean, min, max, etc.) and round the values for readability
df[cols].describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,32.37,64.76,2279.73
std,24.56,30.09,2266.79
min,0.00,18.25,0.00
25%,9.00,35.50,398.55
50%,29.00,70.35,1394.55
75%,55.00,89.85,3786.60
max,72.00,118.75,8684.80


### Cell 2 — apply StandardScaler (mean=0, std=1)

In [3]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()  #it uses the standard scaling for these 3 columns, first with .fit it learns mean and standard deviation of numbers and uses the formula
X = df[cols]
X_std = scaler.fit_transform(X)
X_std_df = pd.DataFrame(X_std, columns=cols) #after scaling, model gives the answers with numpy array, this code makes the column name back
X_std_df.describe().round(2) #it shows all the numerical statistic information of the values in each column,rounded by 2

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,-0.00,-0.00,-0.00
std,1.00,1.00,1.00
min,-1.32,-1.55,-1.01
25%,-0.95,-0.97,-0.83
50%,-0.14,0.19,-0.39
75%,0.92,0.83,0.66
max,1.61,1.79,2.83


### Cell 3 — apply MinMaxScaler (range 0 to 1)

In [4]:
from sklearn.preprocessing import MinMaxScaler
mm = MinMaxScaler()
X_mm = mm.fit_transform(X)   #it learns every column's minimum and maximum values and uses the MinMax formula
X_mm_df = pd.DataFrame(X_mm, columns=cols) #after the formula, it gives the answers as numpy array  which doesnot have a name, this code is for making the names back
X_mm_df.agg(['min', 'max']).round(2) #it shows every columns min and max values rounded by 2, to see if the scaling worked

,tenure,MonthlyCharges,TotalCharges
min,0.0,0.0,0.0
max,1.0,1.0,1.0


### Cell 4 — apply RobustScaler (uses median + IQR, ignores outliers)

In [5]:
from sklearn.preprocessing import RobustScaler
rb = RobustScaler()
X_rb = rb.fit_transform(X)   #it learns the median and IQR value of each column, then it uses its formula, that scaling is much better than other two if it is about outliers
X_rb_df = pd.DataFrame(X_rb, columns=cols)
X_rb_df.describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00
mean,0.07,-0.10,0.26
std,0.53,0.55,0.67
min,-0.63,-0.96,-0.41
25%,-0.43,-0.64,-0.29
50%,0.00,0.00,0.00
75%,0.57,0.36,0.71
max,0.93,0.89,2.15


### Cell 5 — split FIRST, then scale (the big rule — no data leakage)

In [8]:
# Import the function used to split data into training and testing sets
from sklearn.model_selection import train_test_split

# Split the dataset:
# - X is the feature dataset
# - 80% goes to the training set
# - 20% goes to the testing set
# - random_state=42 ensures the same split every time you run the code
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# Import and create a StandardScaler object
scaler = StandardScaler()

# Learn the mean and standard deviation from the TRAINING data,
# then use them to scale the training data
X_train_s = scaler.fit_transform(X_train)

# Scale the TEST data using the SAME mean and standard deviation
# learned from the training data (do NOT use fit_transform here)
X_test_s = scaler.transform(X_test)

# Print the number of rows and columns in the training and testing datasets
print(f'Train shape: {X_train_s.shape}, Test shape: {X_test_s.shape}')

# Calculate and print the mean of each column in the scaled training data.
# After StandardScaler, the mean should be approximately 0.
print(f'Train mean (should be ~0): {X_train_s.mean(axis=0).round(2)}')

Train shape: (5634, 3), Test shape: (1409, 3)
Train mean (should be ~0): [ 0. -0.  0.]


### Cell 6 — save the trained scaler to disk

In [9]:
# Import the joblib library, which is used to save and load Python objects
import joblib

# Save the trained StandardScaler object to a file named "my_scaler.joblib"
# The saved scaler includes the learned mean and standard deviation
joblib.dump(scaler, 'my_scaler.joblib')

# Load the saved StandardScaler object back into the variable "loaded"
loaded = joblib.load('my_scaler.joblib')

# Print a message and display the original means learned from the training data
# (These are the means stored inside the scaler, NOT the scaled means.)
print('Saved and loaded. Learned mean:', loaded.mean_.round(2))

Saved and loaded. Learned mean: [  32.37   64.86 2287.09]


---

## 📤 Submit

Comment every cell. Push to `module-3/class_2/submissions/<YourName>/`.